# Self-driving lab demo: match a hidden color

A **self-driving lab (SDL)** runs experiments by itself: a computer picks the next experiment,
a machine does it, a sensor measures the result, and the computer uses that result to choose
a better experiment next time. This demo follows Baird & Sparks, *"What is a minimal working
example for a self-driving laboratory?"* (Matter, 2022): an RGB LED shines on an 8-channel
light sensor (410 to 670 nm), and the goal is to find the LED color whose spectrum matches a
hidden target.

**How the pieces talk to each other** (one experiment):

| | From | To | Message |
|---|---|---|---|
| 1 | this notebook | MQTT broker (cloud) | "set the LED to R, G, B" |
| 2 | MQTT broker | device (Wokwi tab) | same command; the LED lights up, the sensor reads |
| 3 | device | MQTT broker → this notebook | the 8 sensor readings |

The notebook and the device never connect to each other directly: both connect to the
broker, which passes messages between them, just like the paper's real Pico W.

The device is a **simulated board** running in [Wokwi](https://wokwi.com) in another browser
tab: an ESP32 (Wokwi simulates Wi-Fi only on the ESP32) running the same MicroPython code
and the same messages as the paper's Pico W. Its LED ring changes color every time this
notebook sends a command. Wokwi has no AS7341 sensor part, so the device computes the 8
readings with a simple light model (each LED color adds light to certain channels, plus
room light and 2% noise).

**Before you start:**

1. Open the Wokwi project, press the green ▶ button, and wait until the small screen says
   **Ready!** (up to a minute).
2. Put Wokwi and this notebook **side by side, in two separate browser windows**, not two
   tabs of one window. Wokwi nearly pauses when its tab is hidden, and then the notebook's
   commands time out. If that happens, bring Wokwi back into view: it reconnects within a
   few seconds.
3. Run the cells below one at a time (Shift + Enter).

## Step 0. Load the tools and your settings

**What this cell does:** loads the Python libraries and reads your broker settings from the
`.env` file next to this notebook (see `.env.example`). Nothing secret is printed.

**Check:** the `Device` name printed below must match `PICO_ID` in Wokwi's `config.py`.

In [ ]:
# If a package is missing, run:  %pip install paho-mqtt numpy matplotlib scikit-learn scipy
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display

from sdl_client import CHANNELS, LedLab, load_settings

settings = load_settings(".env")
print("Broker :", settings["host"] or "(missing! add MQTT_HOST to .env)", "port", settings["port"])
print("Login  :", "yes" if settings["username"] else "none (public broker)")
print("Device :", settings["pico_id"], "  <- must match PICO_ID in the Wokwi config.py")

**What this cell does:** defines how the spectra are drawn (8 bars, one per sensor channel,
colored by wavelength). Nothing happens yet; just run it.

In [ ]:
# Plot helpers (nothing to change here)
WAVELENGTHS = [int(c[2:]) for c in CHANNELS]
CHANNEL_COLORS = ["#7b2fbe", "#3b3bff", "#0f8bff", "#00b84a", "#9ccc00", "#ffb300", "#ff6d00", "#d50000"]
METHOD_COLORS = {"Random search": "#8a8a8a", "Grid search": "#e08a00", "Bayesian optimization": "#1f5fd6"}


def swatch(ax, rgb):
    ax.set_ylim(top=ax.get_ylim()[1] * 1.45)  # room for the color square
    sw = ax.inset_axes([0.86, 0.72, 0.12, 0.24])
    sw.set_facecolor(np.array(rgb) / 255)
    sw.set_xticks([]); sw.set_yticks([])
    sw.set_title("LED", fontsize=8)


def spectrum_plot(readings, rgb=None, title="", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 3))
    ax.bar([str(w) for w in WAVELENGTHS], readings, color=CHANNEL_COLORS)
    ax.set_xlabel("Sensor channel (nm)")
    ax.set_ylabel("Counts")
    ax.set_title(title)
    if rgb is not None:
        swatch(ax, rgb)
    return ax

## Step 1. Connect to the broker

**What this cell does:** logs in to the MQTT broker (the cloud "post office" between this
notebook and the device) and starts listening for the device's replies.

**Watch:** nothing changes on the board yet. The notebook and the device have not talked
to each other; they are both just connected to the broker.

In [ ]:
lab = LedLab.from_env(".env")
print("Connected to the broker. Talking to device:", lab.pico_id)

## Step 2. First experiment: is anyone there?

**What this cell does:** sends one command, "LED off" (R = G = B = 0), and waits for the
8 readings to come back. With the LED off, the sensor only sees room light, so the numbers
are small.

**Watch:** the Wokwi screen changes to **Exp #1** with 8 tiny bars, and the serial log
prints the same 8 numbers you see here. That's one full round trip:
notebook → broker → device → broker → notebook.

If this times out, check that the Wokwi simulation is running and says **Ready!**, and that
`PICO_ID` is the same in `.env` and in Wokwi's `config.py`.

In [ ]:
start = time.time()
readings = lab.measure(0, 0, 0)
print(f"Reply in {time.time() - start:.2f} s")
print(dict(zip(CHANNELS, readings)))

## Step 3. Send a color and watch the Wokwi tab

**What this cell does:** sends one color and plots what the sensor measured.

**Watch:** the LED ring turns that color, and the screen's bars match the plot below.
Then change `R`, `G`, `B` (each 0 to 255) and run the cell again.

In [ ]:
R, G, B = 25, 121, 69 # try changing these!

readings = lab.measure(R, G, B)
spectrum_plot(readings, rgb=(R, G, B), title=f"Measured spectrum for RGB = {(R, G, B)}")
plt.show()

**What this cell does:** four experiments in a row: pure red, green, blue and white,
one second apart.

**Watch:** the ring cycles through the four colors. **Question:** which sensor channels light
up for each color? (Red light lands in the 620 nm channel, blue in 440 to 470 nm.)

In [ ]:
colors = {"red": (255, 0, 0), "green": (0, 255, 0), "blue": (0, 0, 255), "white": (255, 255, 255)}
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, (name, rgb) in zip(axes.flat, colors.items()):
    spectrum_plot(lab.measure(*rgb), rgb=rgb, title=name, ax=ax)
    time.sleep(1)  # time to see each color in the Wokwi tab
fig.tight_layout()
plt.show()

## Step 4. The hidden target

**What this cell does:** picks a secret color and measures it once. You only get to see what
the **sensor** saw, never the color itself. Your job: find a color that produces the same
pattern.

**Watch:** the board measures the target with the ring dark, and the screen says
**TARGET (hidden)**, so the color isn't given away.

In [ ]:
TARGET_SEED = 11262026  # change to get a different secret color
_secret_rgb = tuple(int(v) for v in np.random.default_rng(TARGET_SEED).integers(30, 226, size=3))
target = lab.measure(*_secret_rgb, hidden=True)  # the device won't display this color
spectrum_plot(target, title="Target spectrum (RGB hidden)")
plt.show()

## Step 5. Scoring a guess

We need one number that says how far a measurement $m$ is from the target $t$. A bright and
a dim version of the same color give the same *shape* but different heights, so we first
divide each spectrum by its total (turning counts into shares), then average the gaps over the
8 channels:

$$\text{error} = \frac{1}{8}\sum_{i=1}^{8}\left|\frac{m_i}{\sum_j m_j} - \frac{t_i}{\sum_j t_j}\right|$$

0 means a perfect match; lower is better.

**What this cell does:** defines the score, then tries one guess by hand (mid gray).

**Watch:** the ring shows your guess. Edit `guess` and rerun: can you get below 0.01?

In [ ]:
def normalize(readings):
    # Counts -> share of the total signal in each channel
    r = np.asarray(readings, dtype=float)
    return r / r.sum()


def objective(readings, target):
    # Normalized mean absolute error between a measurement and the target
    return float(np.mean(np.abs(normalize(readings) - normalize(target))))


# Try a guess by hand. Can you get the error below 0.01?
guess = (128, 128, 128)
print(f"Error for {guess}: {objective(lab.measure(*guess), target):.4f}")

## Step 6. Three ways to choose the next experiment

- **Random search:** pick any color. No memory of past results.
- **Grid search:** try evenly spaced colors. Systematic, but it never uses what it learns.
- **Bayesian optimization (BO):** after each experiment, fit a model (a *Gaussian process*)
  that predicts the error for every possible color **and how unsure it is**. Then run the color
  with the highest **expected improvement**: one that is either predicted to be good, or
  uncertain enough that it might be. Balancing *exploit what looks good* against
  *explore what is unknown* is the core idea of a self-driving lab.

**What this cell does:** defines the three strategies. Nothing runs on the board yet.
The first run takes about 15 seconds while the machine-learning libraries load.

In [ ]:
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


def suggest_random(rng):
    # Any color, uniformly at random
    return tuple(int(v) for v in rng.integers(0, 256, size=3))


def suggest_grid(i, n_total):
    # The i-th point of an evenly spaced k x k x k grid, with k^3 >= n_total
    k = int(np.ceil(n_total ** (1 / 3) - 1e-9))
    levels = ((np.arange(k) + 0.5) * 256 / k).astype(int)  # k=3 -> 42, 128, 213
    r, g, b = np.unravel_index(i, (k, k, k))
    return (int(levels[r]), int(levels[g]), int(levels[b]))


def suggest_bo(X, y, rng, n_init=5, n_candidates=3000):
    # Bayesian optimization with a Gaussian process and Expected Improvement
    if len(X) < n_init:  # 0. no data yet: start with a few random colors
        return suggest_random(rng)

    Xs = np.array(X) / 255     # 1. data, scaled to 0..1;
    ys = np.log(np.array(y))   #    the model learns log(error)

    kernel = ConstantKernel(1.0) * Matern(  # 2. model: nearby colors give similar errors,
        length_scale=[0.3, 0.3, 0.3], length_scale_bounds=(1e-2, 10), nu=2.5
    ) + WhiteKernel(1e-3, noise_level_bounds=(1e-6, 1e-1))  # and measurements are noisy
    gp = GaussianProcessRegressor(kernel, normalize_y=True, n_restarts_optimizer=3, random_state=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gp.fit(Xs, ys)

    best_x = Xs[np.argmin(ys)]  # 3. candidates: random colors + some near the best so far
    near_best = np.clip(best_x + 0.05 * rng.standard_normal((n_candidates // 3, 3)), 0, 1)
    candidates = np.vstack([rng.random((n_candidates, 3)), near_best])

    mu, sigma = gp.predict(candidates, return_std=True)  # 4. expected improvement
    sigma = np.maximum(sigma, 1e-9)
    improvement = ys.min() - mu
    z = improvement / sigma
    ei = improvement * norm.cdf(z) + sigma * norm.pdf(z)

    return tuple(int(v) for v in np.round(candidates[np.argmax(ei)] * 255))  # 5. best EI


def propose_next(method, X, y, rng, n_total):
    if method == "Random search":
        return suggest_random(rng)
    if method == "Grid search":
        return suggest_grid(len(X), n_total)
    return suggest_bo(X, y, rng)

**What this cell does:** defines the self-driving loop that runs in Step 7. Each turn it
**decides** a color (with the chosen strategy), **runs the experiment** on the board,
**scores** the result, and **redraws** the plot, with no human in the loop. Nothing runs yet.

In [ ]:
def live_figure(X, y, readings, method, n_total):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
    it = np.arange(1, len(y) + 1)
    ax1.scatter(it, y, s=30, c=[np.array(x) / 255 for x in X], edgecolors="0.3", linewidths=0.6,
                zorder=3, label="each experiment")
    ax1.plot(it, np.minimum.accumulate(y), color="black", lw=2, label="best so far")
    ax1.set_yscale("log"); ax1.set_xlim(0.5, n_total + 0.5)
    ax1.set_xlabel("Experiment"); ax1.set_ylabel("Error (log scale)"); ax1.legend(fontsize=8)

    x = np.arange(len(WAVELENGTHS)); w = 0.4
    ax2.bar(x - w / 2, normalize(target), w, color="white", edgecolor="black", label="target")
    ax2.bar(x + w / 2, normalize(readings), w, color=CHANNEL_COLORS, label="this experiment")
    ax2.set_xticks(x, [str(v) for v in WAVELENGTHS])
    ax2.set_xlabel("Sensor channel (nm)"); ax2.set_ylabel("Share of total signal")
    ax2.legend(loc="upper left", fontsize=8)
    swatch(ax2, X[-1])
    fig.suptitle(f"{method}: experiment {len(y)}/{n_total}, RGB = {X[-1]}, error = {y[-1]:.4f}")
    fig.tight_layout()
    return fig


def run_sdl(method, n_experiments=20):
    # The self-driving loop: decide -> experiment -> score -> repeat
    rng = np.random.default_rng()
    X, y = [], []
    for _ in range(n_experiments):
        rgb = propose_next(method, X, y, rng, n_experiments)  # 1. decide
        readings = lab.measure(*rgb)                          # 2. experiment (watch Wokwi!)
        X.append(rgb)
        y.append(objective(readings, target))                 # 3. score
        fig = live_figure(X, y, readings, method, n_experiments)
        clear_output(wait=True)                               # 4. show it live
        display(fig)
        plt.close(fig)
    best = int(np.argmin(y))
    print(f"{method}: best error {y[best]:.4f} with RGB = {X[best]}, found at experiment {best + 1}")
    return {"X": X, "y": y}


runs = {}
N_EXPERIMENTS = 20

## Step 7. Run the self-driving lab

Each cell below runs 20 experiments on its own, about 15 to 30 seconds per strategy. The plot
redraws after every experiment: on the left, each experiment's error (dots colored like the
LED) and the best so far; on the right, the latest spectrum against the target.

### 7a. Random search

**Watch:** the ring jumps between unrelated colors; the error goes down only by luck.

In [ ]:
runs["Random search"] = run_sdl("Random search", N_EXPERIMENTS)

### 7b. Grid search

**Watch:** the colors step through a fixed pattern (low, medium, high for each of red, green,
blue), whatever the results say.

In [ ]:
runs["Grid search"] = run_sdl("Grid search", N_EXPERIMENTS)

### 7c. Bayesian optimization

**Watch:** the first 5 colors are random, to get some data. After that the model picks each
color, and the ring settles on similar shades as it closes in on the target.

In [ ]:
runs["Bayesian optimization"] = run_sdl("Bayesian optimization", N_EXPERIMENTS)

## Step 8. Compare the optimizers

**What this cell does:** puts the three "best so far" curves on one plot, lists each
strategy's best color, and finally reveals the secret color.

**Question:** which strategy reaches a low error in the fewest experiments? Run Step 7 again:
does the ranking hold?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for name, r in runs.items():
    it = np.arange(1, len(r["y"]) + 1)
    ax.plot(it, np.minimum.accumulate(r["y"]), lw=2, label=name, color=METHOD_COLORS.get(name))
ax.set_yscale("log"); ax.set_xlabel("Experiment"); ax.set_ylabel("Best error so far (log scale)")
ax.legend(fontsize=9); fig.tight_layout(); plt.show()

print(f"{'Optimizer':<24}{'Best error':>12}  {'Best RGB':<18}Found at")
for name, r in runs.items():
    i = int(np.argmin(r["y"]))
    print(f"{name:<24}{r['y'][i]:>12.4f}  {str(r['X'][i]):<18}{i + 1}")
print("\nThe secret color was RGB =", _secret_rgb)

## Step 9. Disconnect

**What this cell does:** turns the LED off and logs out of the broker.

**Watch:** the ring goes dark.

In [ ]:
lab.measure(0, 0, 0)  # LED off
lab.close()
print("Disconnected.")